In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# =========================
# CONFIG
# =========================
BASE_DIR = Path("./")          # txt 파일들이 있는 폴더
OUT_DIR = Path("./results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [41, 42, 43, 44, 45]

# 원본 duration 분포 참고 파일
ORIG_DURATION_FILE = BASE_DIR / "MIMICEL_time_duration_norm.txt"

# activity id -> activity name
# act_dict/id2activity 있으면 여기를 실제 이름으로 바꾸면 됨
ID2ACT = {
    "1": "Discharge from the ED",
    "2": "Enter the ED",
    "3": "Medicine dispensations",
    "4": "Medicine reconciliation",
    "5": "Triage in the ED",
    "6": "Vital sign check",
}

BASE_TIMESTAMP = pd.Timestamp("2026-01-01 00:00:00")


# =========================
# FUNCTIONS
# =========================
def read_int_traces(path):
    traces = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                traces.append(line.split())
    return traces


def read_float_traces(path):
    traces = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                traces.append([float(x) for x in line.split()])
    return traces


def read_durations(path):
    vals = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                vals.append(float(line))
    return np.array(vals)


def restore_one_seed(seed):
    act_file = BASE_DIR / f"{seed}_3000_result_trans.txt"
    time_file = BASE_DIR / f"{seed}_3000_result_trans_time.txt"

    act_traces = read_int_traces(act_file)
    time_traces = read_float_traces(time_file)
    orig_durations = read_durations(ORIG_DURATION_FILE)

    rows = []

    for case_idx, (acts, diffs) in enumerate(zip(act_traces, time_traces), start=1):
        n = min(len(acts), len(diffs))
        acts = acts[:n]
        diffs = np.array(diffs[:n], dtype=float)

        # 음수 방어
        diffs = np.clip(diffs, 0, None)

        # trace별 time diff가 전부 0이면 균등 간격 처리
        if diffs.sum() == 0:
            diffs = np.ones(n) / n
        else:
            diffs = diffs / diffs.sum()

        # synthetic case duration은 원본 duration 분포에서 순환 사용
        duration_norm = orig_durations[(case_idx - 1) % len(orig_durations)]

        # 보기 좋게 시간 단위로 환산
        # duration_norm 자체가 정규화값이므로, 여기서는 상대 시간 복원용
        rel_times = np.cumsum(diffs) * duration_norm

        case_id = f"syn_seed{seed}_case{case_idx:04d}"
        case_start = BASE_TIMESTAMP + pd.Timedelta(minutes=case_idx)

        for event_idx, (act_id, rel_t) in enumerate(zip(acts, rel_times), start=1):
            rows.append({
                "stay_id": case_id,
                "event_order": event_idx,
                "activity_id": int(act_id),
                "activity": ID2ACT.get(str(act_id), f"activity_{act_id}"),
                "time_diff_norm": float(diffs[event_idx - 1]),
                "case_duration_norm": float(duration_norm),
                "relative_time_norm": float(rel_t),
                "timestamps": case_start + pd.to_timedelta(rel_t, unit="D")
            })

    df = pd.DataFrame(rows)
    out_csv = OUT_DIR / f"processgan_restored_synthetic_seed{seed}.csv"
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")

    print(f"[OK] seed {seed}: {len(df):,} events -> {out_csv}")
    return df


# =========================
# RUN
# =========================
all_dfs = []

for seed in SEEDS:
    df_seed = restore_one_seed(seed)
    all_dfs.append(df_seed)

df_all = pd.concat(all_dfs, ignore_index=True)
df_all.to_csv(OUT_DIR / "processgan_restored_synthetic_all_seeds.csv",
              index=False, encoding="utf-8-sig")

print("[DONE]", df_all.shape)

[OK] seed 41: 10,355 events -> results\processgan_restored_synthetic_seed41.csv
[OK] seed 42: 19,578 events -> results\processgan_restored_synthetic_seed42.csv
[OK] seed 43: 22,376 events -> results\processgan_restored_synthetic_seed43.csv
[OK] seed 44: 17,590 events -> results\processgan_restored_synthetic_seed44.csv
[OK] seed 45: 15,192 events -> results\processgan_restored_synthetic_seed45.csv
[DONE] (85091, 8)
